# Deploy llm-d on Kubernetes in 15 Minutes

This notebook is a fast-paced quickstart that takes you from zero to serving
LLM inference requests with llm-d. We will deploy a small model (Qwen3-0.6B)
so everything starts quickly and you can see the full system in action.

**Total time: ~15 minutes** (with a running Kubernetes cluster)

| Step | What | Time |
|------|------|------|
| 1 | Prerequisites check | 1 min |
| 2 | Clone repo | 30 sec |
| 3 | Deploy gateway prerequisites | 2 min |
| 4 | Deploy model server | 5 min |
| 5 | Deploy router | 2 min |
| 6 | Verify and send first request | 2 min |
| 7 | Check metrics | 1 min |
| 8 | Cleanup | 1 min |

## Step 1: Prerequisites Check (~1 minute)

Before we begin, let's verify you have everything you need:

- A Kubernetes cluster with at least one GPU node (any NVIDIA GPU works for Qwen3-0.6B)
- `kubectl` configured and pointing at your cluster
- `helm` v3.x installed
- A Hugging Face token (free account is fine for Qwen3-0.6B)

Run the cell below to check all tools at once.

In [ ]:
import subprocess
import sys
import shutil

checks_passed = True

# Check kubectl
if shutil.which("kubectl"):
    result = subprocess.run(["kubectl", "version", "--client", "--short"], capture_output=True, text=True)
    print(f"[OK] kubectl: {result.stdout.strip()}")
else:
    print("[FAIL] kubectl not found. Install from https://kubernetes.io/docs/tasks/tools/")
    checks_passed = False

# Check helm
if shutil.which("helm"):
    result = subprocess.run(["helm", "version", "--short"], capture_output=True, text=True)
    print(f"[OK] helm: {result.stdout.strip()}")
else:
    print("[FAIL] helm not found. Install from https://helm.sh/docs/intro/install/")
    checks_passed = False

# Check cluster connectivity
result = subprocess.run(["kubectl", "cluster-info"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"[OK] Cluster reachable")
else:
    print("[FAIL] Cannot reach Kubernetes cluster. Check your kubeconfig.")
    checks_passed = False

# Check for GPU nodes
result = subprocess.run(
    ["kubectl", "get", "nodes", "-o", "jsonpath={.items[*].status.allocatable.nvidia\\.com/gpu}"],
    capture_output=True, text=True
)
gpu_counts = [g for g in result.stdout.strip().split() if g and g != "0"]
if gpu_counts:
    print(f"[OK] GPU nodes found: {len(gpu_counts)} node(s) with GPUs")
else:
    print("[WARN] No GPU nodes detected. Model server will not schedule without GPUs.")

if checks_passed:
    print("\nAll prerequisites met. Let's go!")
else:
    print("\nPlease fix the issues above before continuing.")

## Step 2: Clone the Repository (~30 seconds)

The llm-d production stack repository contains all the deployment manifests,
guides, and recipes you need. We will use the `optimized-baseline` guide
with a small model for this quickstart.

In [ ]:
import os

# Clone the production stack repo
!git clone https://github.com/llm-d/llm-d-production-stack.git 2>/dev/null || \
    echo "Repository already cloned."

# Set environment variables for the entire deployment
os.environ["NAMESPACE"] = "llm-d-quickstart"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-0.6B"  # Small model for fast startup
os.environ["HF_TOKEN"] = "<your-huggingface-token>"  # Replace with your token

print(f"Namespace:  {os.environ['NAMESPACE']}")
print(f"Model:      {os.environ['MODEL_NAME']}")
print(f"Repo:       llm-d-production-stack/")
print()
print("Using Qwen3-0.6B because it is small enough to load in under a minute")
print("on most GPUs. For production, swap in Qwen3-32B, Llama-3.1-70B, etc.")

## Step 3: Deploy Gateway Prerequisites (~2 minutes)

llm-d uses the Kubernetes Gateway API with Envoy Gateway as the data plane.
This gives you a production-grade ingress that exposes an OpenAI-compatible API.

**This step takes about 2 minutes** while the Envoy Gateway controller starts up.

In [ ]:
import time

start = time.time()

# Create the namespace
!kubectl create namespace $NAMESPACE --dry-run=client -o yaml | kubectl apply -f -

# Install Gateway API CRDs (standard Kubernetes Gateway API)
print("\nInstalling Gateway API CRDs...")
!kubectl apply -f https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.2.1/standard-install.yaml

# Install Envoy Gateway controller
print("\nInstalling Envoy Gateway...")
!helm install eg oci://docker.io/envoyproxy/gateway-helm \
    --version v1.3.0 \
    -n envoy-gateway-system \
    --create-namespace \
    2>/dev/null || echo "Envoy Gateway already installed."

# Wait for the Envoy Gateway controller to be ready
print("\nWaiting for Envoy Gateway controller...")
!kubectl wait --for=condition=available deployment/envoy-gateway \
    -n envoy-gateway-system --timeout=120s

elapsed = time.time() - start
print(f"\nGateway prerequisites ready in {elapsed:.0f} seconds.")

## Step 4: Deploy the Model Server (~5 minutes)

Now we deploy vLLM serving Qwen3-0.6B. This is the component that actually
runs inference on the GPU.

**This step takes about 5 minutes.** Most of the time is spent pulling the
container image and downloading model weights. With Qwen3-0.6B (only ~1.2 GB),
this is much faster than a full-size model like Qwen3-32B (~60 GB).

In [ ]:
import time

start = time.time()

# Create the Hugging Face token secret
!kubectl create secret generic hf-token \
    --from-literal=token=$HF_TOKEN \
    -n $NAMESPACE \
    --dry-run=client -o yaml | kubectl apply -f -

# Deploy the model server using the optimized-baseline guide
# We override the model name to use the smaller Qwen3-0.6B
print("\nDeploying vLLM model server with Qwen3-0.6B...")
!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/model-server/ \
    -n $NAMESPACE

# Patch the deployment to use our smaller model
!kubectl set env deployment/vllm -n $NAMESPACE \
    MODEL_NAME=Qwen/Qwen3-0.6B

print("\nModel server deployment submitted.")
print("Waiting for the model to download and load into GPU memory...")
print("(This takes about 3-5 minutes depending on network speed)")

# Wait for the model server pod to become ready
!kubectl wait --for=condition=ready pod -l app=vllm \
    -n $NAMESPACE --timeout=600s

elapsed = time.time() - start
print(f"\nModel server ready in {elapsed:.0f} seconds!")

## Step 5: Deploy the Router (~2 minutes)

The router is the brain of llm-d. It consists of two components:

- **Envoy Proxy** - handles HTTP traffic and the OpenAI-compatible API
- **EPP (Endpoint Picker)** - an ext-proc filter that picks the best backend
  replica for each request based on prefix-cache affinity and load

**This step takes about 2 minutes.**

In [ ]:
import time

start = time.time()

# Deploy the router (EPP + Envoy configuration)
print("Deploying the llm-d router...")
!kubectl apply -k llm-d-production-stack/guides/optimized-baseline/router/ \
    -n $NAMESPACE

# Wait for the EPP to be ready
print("\nWaiting for EPP router...")
!kubectl wait --for=condition=ready pod -l app=epp \
    -n $NAMESPACE --timeout=120s

# Wait for the Envoy proxy to be ready
print("\nWaiting for Envoy proxy...")
!kubectl wait --for=condition=ready pod -l app=envoy \
    -n $NAMESPACE --timeout=120s

elapsed = time.time() - start
print(f"\nRouter ready in {elapsed:.0f} seconds!")

## Step 6: Verify Pods and Send Your First Request (~2 minutes)

Everything should be running now. Let's verify and send a test request.

In [ ]:
# Verify all pods are running
print("=== All Pods in the llm-d-quickstart Namespace ===")
!kubectl get pods -n $NAMESPACE -o wide

print("\n=== Services ===")
!kubectl get svc -n $NAMESPACE

print("\n=== InferencePool ===")
!kubectl get inferencepool -n $NAMESPACE 2>/dev/null || echo "(InferencePool CRD not installed)"

# Count pods by status
print("\n=== Pod Summary ===")
!kubectl get pods -n $NAMESPACE --no-headers | awk '{print $3}' | sort | uniq -c

In [ ]:
# Send your first request!
import subprocess
import json
import time

# Get the gateway IP
gateway_ip = subprocess.check_output(
    "kubectl get svc -n $NAMESPACE envoy-gateway "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway endpoint: http://{gateway_ip}:8080")
print()

# Send a simple chat completion request
print("Sending your first request to llm-d...")
print()

start = time.time()
result = subprocess.run(
    ["curl", "-s", f"http://{gateway_ip}:8080/v1/chat/completions",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "Qwen/Qwen3-0.6B",
         "messages": [{"role": "user", "content": "What is Kubernetes in one sentence?"}],
         "max_tokens": 100
     })],
    capture_output=True, text=True
)
elapsed = time.time() - start

try:
    response = json.loads(result.stdout)
    print("Response:")
    print(json.dumps(response, indent=2))
    print(f"\nLatency: {elapsed:.3f}s")
    print("\nYou just served your first LLM request through llm-d!")
except json.JSONDecodeError:
    print(f"Raw response: {result.stdout}")
    print(f"Errors: {result.stderr}")

In [ ]:
# Send a second request with the same prefix to demonstrate cache hits
print("Sending a second request with the same system prompt...")
print("The router should route it to the same replica for a prefix-cache hit.")
print()

start = time.time()
result = subprocess.run(
    ["curl", "-s", f"http://{gateway_ip}:8080/v1/chat/completions",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "Qwen/Qwen3-0.6B",
         "messages": [
             {"role": "system", "content": "You are a helpful assistant that explains technology concepts simply."},
             {"role": "user", "content": "What is Kubernetes?"}
         ],
         "max_tokens": 100
     })],
    capture_output=True, text=True
)
first_latency = time.time() - start

start = time.time()
result2 = subprocess.run(
    ["curl", "-s", f"http://{gateway_ip}:8080/v1/chat/completions",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "Qwen/Qwen3-0.6B",
         "messages": [
             {"role": "system", "content": "You are a helpful assistant that explains technology concepts simply."},
             {"role": "user", "content": "What is a container?"}
         ],
         "max_tokens": 100
     })],
    capture_output=True, text=True
)
second_latency = time.time() - start

print(f"First request (cold):  {first_latency:.3f}s")
print(f"Second request (warm): {second_latency:.3f}s")
print()
print("The second request shares the same system prompt prefix. The EPP routes")
print("it to the replica that already has that prefix cached, so the prefill")
print("phase is faster (it only processes the new user message).")

## Step 7: Check Metrics (~1 minute)

The EPP exposes Prometheus metrics on port 9090. Let's take a quick look at
what is available.

In [ ]:
# Port-forward the EPP metrics endpoint and check key metrics
# In a separate terminal, run:
#   kubectl port-forward svc/epp -n llm-d-quickstart 9090:9090

import urllib.request

EPP_METRICS = "http://localhost:9090/metrics"

try:
    with urllib.request.urlopen(EPP_METRICS, timeout=5) as resp:
        metrics = resp.read().decode("utf-8")

    # Show key metrics
    key_prefixes = [
        "epp_request_total",
        "epp_queue_depth",
        "epp_prefix_cache_hit",
        "epp_request_ttft",
    ]

    print("=== Key EPP Metrics ===")
    for line in metrics.split("\n"):
        if line.startswith("#"):
            continue
        for prefix in key_prefixes:
            if line.startswith(prefix):
                print(f"  {line}")
                break

except Exception as e:
    print(f"Could not reach metrics endpoint: {e}")
    print()
    print("To view metrics, port-forward the EPP service:")
    print(f"  kubectl port-forward svc/epp -n {os.environ['NAMESPACE']} 9090:9090")
    print()
    print("Then re-run this cell, or open http://localhost:9090/metrics in your browser.")

In [ ]:
# Also check vLLM server metrics
# The model server exposes its own metrics including GPU utilization and KV-cache usage

print("=== vLLM Model Server Logs (last 10 lines) ===")
!kubectl logs deployment/vllm -n $NAMESPACE --tail=10

print("\n=== EPP Router Logs (last 10 lines) ===")
!kubectl logs deployment/epp -n $NAMESPACE --tail=10

## Step 8: Cleanup (~1 minute)

When you are done experimenting, clean up all the resources.

**Warning:** This deletes everything in the quickstart namespace. Skip this
cell if you want to keep experimenting.

In [ ]:
# Uncomment the lines below to clean up

# Delete all resources in the quickstart namespace
# !kubectl delete namespace $NAMESPACE

# Optionally remove Envoy Gateway (only if no other workloads use it)
# !helm uninstall eg -n envoy-gateway-system
# !kubectl delete namespace envoy-gateway-system

# Remove the cloned repo
# !rm -rf llm-d-production-stack

print("Cleanup commands are commented out for safety.")
print("Uncomment the lines above and re-run to delete all resources.")

## What You Deployed

In 15 minutes, you set up a production-grade LLM inference stack:

```
Client (curl / OpenAI SDK)
       |
       v
+--------------+
| Envoy Gateway|  OpenAI-compatible API on port 8080
+------+-------+
       |
       v
+------+-------+
| EPP Router   |  Prefix-cache-aware + load-aware routing
+------+-------+
       |
       v
+------+-------+
| vLLM Server  |  Qwen3-0.6B on GPU with automatic prefix caching
+--------------+
```

### What to Do Next

Now that you have a running system, explore these topics:

- **Scale up the model** - swap Qwen3-0.6B for Qwen3-32B or Llama-3.1-70B
- **Add more replicas** - scale the vLLM deployment to multiple replicas and
  watch the EPP distribute requests intelligently
- **Enable P/D disaggregation** - split prefill and decode for higher throughput
- **Set up observability** - deploy Prometheus and Grafana dashboards
- **Configure flow control** - add priority bands and multi-tenant fairness

Check out the other notebooks in this series for detailed walkthroughs of each.